In [2]:
"""
===============================
Convert a MOABB dataset to BIDS
===============================

The Brain Imaging Data Structure (BIDS) format
is standard for storing neuroimaging data.
It follows fixed principles to facilitate the
sharing of neuroimaging data between researchers.

The MOABB library allows to convert any MOABB dataset to
BIDS [1]_ and [2]_.

In this example, we will convert the AlexMI dataset to BIDS using the
option ``cache_config=dict(path=temp_dir, save_raw=True)`` of the ``get_data``
method from the dataset object.

This will automatically save the raw data in the BIDS format and allow to use
a cache for the next time the dataset is used.

We will use the AlexMI dataset [3]_, one of the smallest in
people and one that can be downloaded quickly.
"""

# Authors: Pierre Guetschel <pierre.guetschel@gmail.com>
#
# License: BSD (3-clause)

import shutil
import tempfile
from pathlib import Path

import mne

from moabb import set_log_level
from moabb.datasets import AlexMI


set_log_level("info")

In [3]:

###############################################################################
# Basic usage
# -----------
#
# Here, we will save the BIDS version of the dataset in a temporary folder
temp_dir = Path(tempfile.mkdtemp())
# The conversion of any MOABB dataset to a BIDS-compliant structure can be done
# by simply calling its ``get_data`` method and using the ``cache_config``
# parameter. This parameter is a dictionary.
dataset = AlexMI()
# Reducing the number of subjects to speed up the example

dataset.subject_list = dataset.subject_list[:1]
_ = dataset.get_data(cache_config=dict(path=temp_dir, save_raw=True))

/Users/sopsahl/nta/thesis/projects/moabb/moabb/datasets/download.py:60: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_ALEXEEG_PATH"
  set_config(key, get_config("MNE_DATA"))
/Users/sopsahl/miniconda3/envs/moabb/lib/python3.14/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'zenodo.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/sopsahl/miniconda3/envs/moabb/lib/python3.14/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'zenodo.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
100%|█████████████████████████████████████| 17.3M/17.3M [00:00<00:00, 34.7GB/s]
SHA256 hash of downloaded file: 8e2897c11600687e34e3719d41c36936

In [4]:
###############################################################################
# Before / after folder structure
# -------------------------------
#
# To investigate what was saved, we will first define a function to print
# the folder structure of a given path:
def print_tree(p: Path, last=True, header=""):
    elbow = "└──"
    pipe = "│  "
    tee = "├──"
    blank = "   "
    print(header + (elbow if last else tee) + p.name)
    if p.is_dir():
        children = list(p.iterdir())
        for i, c in enumerate(children):
            print_tree(
                c, header=header + (blank if last else pipe), last=i == len(children) - 1
            )


In [5]:
###############################################################################
# Now, we will retrieve the location of the original dataset. It is stored
# in the MNE data directory, which can be found with the ``"MNE_DATA"`` key:
mne_data = Path(mne.get_config("MNE_DATA"))
print(f"MNE data directory: {mne_data}")

###############################################################################
# Now, we can print the folder structure of the original dataset:
print("Before conversion:")
print_tree(mne_data / "MNE-alexeeg-data")

MNE data directory: /Users/sopsahl/mne_data
Before conversion:
└──MNE-alexeeg-data
   └──record
      └──806023
         └──files
            └──subject1.raw.fif


In [6]:
###############################################################################
# As we can see, before conversion, all the data (i.e. from all subjects,
# sessions and runs) is stored in a single folder. This follows no particular
# standard and can vary from one dataset to another.
#
# After conversion, the data is stored in a BIDS-compliant way:
print("After conversion:")
print_tree(temp_dir / "MNE-BIDS-alexandre-motor-imagery")

After conversion:
└──MNE-BIDS-alexandre-motor-imagery
   ├──sub-1
   │  ├──sub-1_desc-846a5ffe84a46e90b4b0f355596dfe11_lockfile.json
   │  └──ses-0
   │     ├──sub-1_ses-0_scans.tsv
   │     └──eeg
   │        ├──sub-1_ses-0_task-imagery_run-0_desc-846a5ffe84a46e90b4b0f355596dfe11_events.tsv
   │        ├──sub-1_ses-0_task-imagery_run-0_desc-846a5ffe84a46e90b4b0f355596dfe11_events.json
   │        ├──sub-1_ses-0_task-imagery_run-0_desc-846a5ffe84a46e90b4b0f355596dfe11_eeg.edf
   │        ├──sub-1_ses-0_task-imagery_run-0_desc-846a5ffe84a46e90b4b0f355596dfe11_eeg.json
   │        └──sub-1_ses-0_task-imagery_run-0_desc-846a5ffe84a46e90b4b0f355596dfe11_channels.tsv
   ├──README
   ├──dataset_description.json
   ├──participants.json
   └──participants.tsv


In [7]:
###############################################################################
# In the BIDS version of our dataset, the raw files are saved in EDF.
# The data is organized in a hierarchy of folders,
# starting with the subjects, then the sessions, and then the runs. Metadata
# files are stored to describe the data. For more details on the BIDS
# structure, please refer to the `BIDS website <https://bids.neuroimaging.io>`_
# and the `BIDS spec <https://bids-specification.readthedocs.io/en/stable/>`_.
#
# Under the hood, saving datasets to BIDS is done through the caching system
# of MOABB. Only raw EEG files are officially supported by the BIDS
# specification.
# However, MOABB's caching mechanism also offers the possibility to save
# the data in a pseudo-BIDS after different preprocessing steps.
# In particular, we can save :class:`mne.Epochs` and ``np.ndarray`` objects.
# For more details on the caching system,
# please refer to the tutorial :doc:`./plot_disk_cache`.
#
# Cleanup
# -------
#
# Finally, we can delete the temporary folder:
shutil.rmtree(temp_dir)